In [ ]:
!pip install datasets==2.19.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 10.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


**Load Dataset**

In [ ]:
from datasets import load_dataset

dataset = load_dataset("conll2003")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/datasets/load.py:1486: FutureWarning: The repository for conll2003 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/conll2003
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


**Understand Labels**

In [ ]:
label_list = dataset["train"].features["ner_tags"].feature.names
print(label_list)

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


**Tokenizer**

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

**Tokenization + Label Alignment**

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)  # special tokens
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])  # first subword
            else:
                label_ids.append(-100)  # other subwords ignored

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs
# preprocessing
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

**Load Model**

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list)
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

**Evaluation Metrics**

In [ ]:
!pip install evaluate seqeval
import evaluate

metric = evaluate.load("seqeval")
def compute_metrics(p):
    predictions, labels = p
    predictions = predictions.argmax(axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=c862df4b324b02e00b1d857c2d83127021a441da51c8c9ff491f6c47e61aa06d
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


**Training Setup**

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    logging_steps=50,
    report_to="none"
)

**Trainer + Training**

In [ ]:
from transformers import Trainer
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].shuffle(seed=42).select(range(5000)),
    eval_dataset=tokenized_datasets["validation"].select(range(1000)),
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()

Step,Training Loss
50,0.758033
100,0.253303
150,0.186449
200,0.138832
250,0.129514
300,0.119716


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=313, training_loss=0.2580965445064508, metrics={'train_runtime': 58.138, 'train_samples_per_second': 86.002, 'train_steps_per_second': 5.384, 'total_flos': 120555254661408.0, 'train_loss': 0.2580965445064508, 'epoch': 1.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 0.085935577750206,
 'eval_precision': 0.8469769930444088,
 'eval_recall': 0.9004550625711035,
 'eval_f1': 0.872897711607389,
 'eval_runtime': 6.3929,
 'eval_samples_per_second': 156.424,
 'eval_steps_per_second': 9.855,
 'epoch': 1.0}

**Analysis**

The fine-tuned BERT model for token classification achieved strong performance, with a precision of 0.85, recall of 0.90, and an F1 score of 0.87. The high recall indicates that the model is highly effective at identifying most of the relevant entities in the dataset, while the precision value suggests that the majority of predicted labels are correct.

The F1 score of 0.87 reflects a good balance between precision and recall, demonstrating that the model performs reliably across both metrics. Overall, the results indicate that the model has successfully learned meaningful token-level representations and is well-suited for chunking tasks.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

sentence = "John works at Google in California"

tokens = tokenizer(sentence.split(), return_tensors="pt", is_split_into_words=True)

tokens = {key: val.to(device) for key, val in tokens.items()}

model.to(device)

outputs = model(**tokens)
predictions = outputs.logits.argmax(dim=2)

predicted_labels = [label_list[p.item()] for p in predictions[0]]

print(list(zip(sentence.split(), predicted_labels)))

[('John', 'O'), ('works', 'B-PER'), ('at', 'O'), ('Google', 'O'), ('in', 'B-ORG'), ('California', 'O')]


The model was tested on a custom sentence to evaluate its real-world performance. While the model was able to assign labels to each token, some predictions were not perfectly accurate. This can be attributed to the limited training data and the small number of training epochs.

Token classification is a challenging task, especially when dealing with limited training samples, and slight inconsistencies in predictions are expected. Despite this, the model demonstrates the ability to perform token-level classification and identify patterns in the text.


# **POS vs Chunking**

Part-of-Speech (POS) tagging and chunking are both token classification tasks but differ in their level of analysis. POS tagging focuses on assigning grammatical labels to individual words, such as nouns, verbs, and adjectives. It provides word-level syntactic information.

In contrast, chunking (also known as shallow parsing) groups words into meaningful phrases such as noun phrases (NP) and verb phrases (VP). It provides a higher-level understanding of sentence structure compared to POS tagging.

While POS tagging is relatively simpler and focuses on individual tokens, chunking is more complex as it captures relationships between words and identifies phrase-level structures.


# Insights

One of the main challenges faced during this project was handling label alignment with subword tokenization. Since BERT splits words into smaller subwords, it was necessary to ensure that labels were correctly mapped to the corresponding tokens while ignoring special tokens.

Another challenge was managing variable-length sequences, which required the use of a data collator for proper padding. Despite these challenges, the model was successfully trained and achieved strong performance, demonstrating the effectiveness of transformer-based models for token classification tasks.
